# Proyecto Loti Perú — Pipeline MLOps
> Notebooks listos para Databricks. Ajustá la variable `DATA_PATH` para tu ruta.

## 01 · Ingesta y Limpieza (Bronze → Silver)

In [0]:
# Configuración
DATA_PATH = "dbfs:/datasets/loterias/apuestas_sample.csv"  # ← ajusta tu ruta (DBFS o UC Volumes)
CATALOG = "main"
SCHEMA  = "loterias_silver"
TABLE_BRONZE = f"{CATALOG}.{SCHEMA}.apuestas_bronze"
TABLE_SILVER = f"{CATALOG}.{SCHEMA}.apuestas_silver"

# Spark session (Databricks la provee; local fallback)
try:
    spark
except NameError:
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.appName("loti-ingesta").getOrCreate()

from pyspark.sql import functions as F, Window

# Crear catálogo/esquema si no existen
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")

# Ingesta CSV → Bronze (Delta)
df_raw = (spark.read.option("header", True).option("inferSchema", True).csv(DATA_PATH))

(df_raw.write.format("delta").mode("overwrite").option("overwriteSchema","true")
 .saveAsTable(TABLE_BRONZE))

display(spark.table(TABLE_BRONZE).limit(5))
print("Bronze OK:", TABLE_BRONZE)

### Controles básicos de calidad

In [0]:
df = spark.table(TABLE_BRONZE)

# 1) Duplicados por tx_id
dups = (df.groupBy("tx_id").count().filter("count > 1"))
dups_count = dups.count()

# 2) Nulos en campos clave
campos_clave = ["tx_id","user_id","monto","min_antes_cierre","ip"]
nulos = {c: df.filter(F.col(c).isNull()).count() for c in campos_clave}

# 3) Rangos razonables
df_clean = (df
    .dropDuplicates(["tx_id"])
    .filter((F.col("monto") > 0) & (F.col("monto") < 10000))
    .withColumn("fecha", F.to_date("fecha"))
)

display(df_clean.limit(5))
print("Duplicados tx_id:", dups_count)
print("Nulos por columna:", nulos)

### Silver (limpio y tipado)

In [0]:
(df_clean.write.format("delta").mode("overwrite").option("overwriteSchema","true")
 .saveAsTable(TABLE_SILVER))

display(spark.table(TABLE_SILVER).limit(10))
print("Silver OK:", TABLE_SILVER)